<a href="https://colab.research.google.com/github/Existanze54/sirius-machine-learning-2026/blob/main/Homeworks/HW5_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашка 5. Случайный лес

### imports

In [ ]:
%pip -q install datamol==0.13.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import datamol as dm
from rdkit.Chem.Draw import IPythonConsole

from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score, auc, precision_recall_curve
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

SEED = 42

## Задача 1. Проницаемость гематоэнцефалического барьера

Поработаем с датасетом [BBBP](https://pubs.acs.org/doi/10.1021/ci300124c). Для каждой молекулы известны ее SMILES и бинарная метка `p_np` — проникает ли вещество через гематоэнцефалический барьер.

In [ ]:
url = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv'
df = pd.read_csv(url)
df = df[['name', 'smiles', 'p_np']]
df.head(2)

### Подготовка данных

Рассчитаем дескрипторы с помощью уже знакомого `datamol`.

In [ ]:
df['mol'] = df['smiles'].map(dm.to_mol)
df = df.dropna()

y = df['p_np']
X = dm.descriptors.batch_compute_many_descriptors(
    df['mol'], progress=True, batch_size=16, n_jobs=-1,
)

Для каждой молекулы выделим **каркас Бемиса — Мурко** (англ. *scaffold*). Его основу составляют кольца и цепочки атомов, которые соединяют эти кольца. Боковые заместители при этом отбрасываются. Например, у фенилаланина и тирозина один и тот же каркас — бензольное кольцо. Если в молекуле нет колец, её каркас будет пустым.

In [ ]:
df['scaffold'] = df['mol'].map(dm.to_scaffold_murcko)
df['scaffold'] = df['scaffold'].map(dm.remove_stereochemistry)
df['scaffold_smiles'] = df['scaffold'].map(dm.to_smiles)

Визуализируем три самых частых каркаса с примерами молекул, которые его содержат.

In [ ]:
counts = df['scaffold_smiles'].value_counts()
top_scaffolds = counts[(counts.index != '') & (counts >= 3)].head(3).index

mols = []
for scaffold in top_scaffolds:
    examples = df[df['scaffold_smiles'] == scaffold].head(3)
    mols.extend([examples['scaffold'].iloc[0], *examples['mol']])

dm.viz.to_image(mols, n_cols=4, mol_size=(240, 180))

### Разбиение

Сначала отложите около 20% молекул для тестирования. Выбирайте молекулы группами: если в тестовую выборку попала молекула с некоторым каркасом, туда же должны попасть все остальные молекулы с этим каркасом.

Из оставшихся молекул сформируйте две валидационные выборки. Для первой отберите около 15% молекул по тому же правилу. Для второй случайно отберите ещё 15%, не учитывая каркасы. Все проценты считайте от исходного числа молекул.

Оставшиеся молекулы используйте для обучения. Получится четыре непересекающиеся выборки: одна обучающая, две валидационные и одна общая тестовая.

In [ ]:
# your code here

### Отбор признаков

Самостоятельно реализуйте **рекурсивное исключение признаков** (*recursive feature elimination*, RFE). Начните со всех дескрипторов. На каждом шаге обучайте `RandomForestClassifier` с параметрами по умолчанию на обучающей выборке и вычисляйте $AUROC$ на обеих валидационных выборках. Затем исключайте дескриптор с наименьшей важностью по `feature_importances_` и переобучайте модель. Продолжайте, пока не оцените модель с одним дескриптором.

На одном графике покажите, как $AUROC$ зависит от числа оставшихся дескрипторов на случайной валидации и на валидации с разбиением по каркасам. Для каждой выборки выберите набор дескрипторов, на котором достигнута максимальная $AUROC$.

In [ ]:
# your code here

### Тестирование

Объедините обучающую и обе валидационные выборки. На объединённой выборке обучите два случайных леса, по одному для каждого набора дескрипторов, выбранного на предыдущем шаге. Вычислите $AUROC$ и $AUPRC$ обеих моделей на одной и той же тестовой выборке. Какой набор дескрипторов дал лучшее качество на молекулах с новыми каркасами?

In [ ]:
# your code here

## Задача 2. Ушки, кольца и ансамбли

Возраст морского ушка определяют по числу колец на срезе раковины. В наборе данных [Abalone](https://archive.ics.uci.edu/dataset/1/abalone) число колец известно для 4 177 моллюсков. Попробуем предсказать его по геометрическим характеристикам, массе, а также полу особи.

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data'

columns = ['sex', 'length', 'diameter', 'height', 'whole_weight',
           'shucked_weight', 'viscera_weight', 'shell_weight', 'rings']

df = pd.read_csv(url, names=columns)
df.head(2)

Закодируем единственную категориальную переменную и разделим на обучающую и валидационные выборки.

In [ ]:
df['is_female'] = df['sex'] == 'F'
df['is_infant'] = df['sex'] == 'I'

X = df.drop(columns=['sex', 'rings'])
y = df['rings']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,
                                                  random_state=SEED)

Обучите три ансамбля по 50 деревьев без ограничения глубины:

1. **Бэггинг.** Используйте `BaggingRegressor` со всеми признаками у каждого дерева;
2. **Случайные подпространства.** Используйте `BaggingRegressor` с шестью признаками на дерево;
3. **Случайный лес.** Используйте `RandomForestRegressor` с тремя признаками-кандидатами на разбиение узла.

Извлеките обученные деревья из каждого ансамбля через `estimators_`, примените их к валидационной выборке и сохраните прогнозы каждого дерева. Для `BaggingRegressor` используйте `estimators_features_`, чтобы выбрать нужные столбцы для каждого дерева. Усредните прогнозы деревьев в каждом ансамбле.

Используя сохраненные предсказания, сравните три показателя для каждого ансамбля:

1. Среднюю $MAE$ и $R^2$ отдельных деревьев;
2. Среднюю попарную корреляцию остатков деревьев;
3. $MAE$ и $R^2$ полного ансамбля.

Объясните результат.

In [ ]:
# your code here